# 🚀 ADK 冒險之旅 — 多代理編排 **（ADK 2.0 版）**

## 用一張 `Workflow` 圖取代 `SequentialAgent` / `LoopAgent` / `ParallelAgent`

---

## 為什麼有這一版

舊版這本教材的主題，正好是 ADK 2.0 **已經 deprecated** 的三個類別。
把它們 import 進來就會看到：

```
DeprecationWarning: SequentialAgent is deprecated in favor of Workflow
                    and will be removed in a future version.
```

它們現在還能跑（我實測過，ADK 2.8 全部正常），但教材如果繼續教這三個，
學生學到的是即將消失的 API。這一版把三種編排全部改寫成 `Workflow`，
而且因為 `Workflow` 有一張真正的圖，**每一種編排都可以畫出來**。

| 主題 | 舊版 | 這一版 |
|---|---|---|
| 順序執行 | `SequentialAgent(sub_agents=[a, b])` | `Workflow(edges=[(START, a), (a, b)])` |
| 並行執行 | `ParallelAgent(sub_agents=[a,b,c])`，外面再包一層 `SequentialAgent` 才能接整合 | 一張圖：fan-out ＋ **`JoinNode`** ＋ 整合，全部畫在一起 |
| 迭代改進 | `LoopAgent(max_iterations=3)` | routing map **回頭邊**（⚠️ 沒有內建 `max_iterations`，要自己守，見 §3） |
| 看得到流程 | ❌ 只有 markdown 手畫 ASCII | ✅ `plot_workflow_graph()` 真的畫出來，還能用執行狀態上色 |
| 條件分支 | ❌ 得在 Python 端寫 if/else | ✅ `(node, {"a": x, "b": y})` ＋ `ctx.route` |

---

## 順便修掉舊版的 4 個問題

1. **`{session.query}` 完全沒有作用**。舊版 `refiner_agent` 的 instruction 裡有這個 placeholder，
   但 ADK 只替換「單層 state key」；帶點的名字會**原封不動當文字**送給模型。
   實測送進 prompt 的就是 `{session.query}` 這 15 個字元。§3 會示範正確做法。
2. **`PROJECT_ID` 寫死在教材裡**。改成從環境變數讀，教材本身不含任何人的專案 ID。
3. **有一格重複的 PLACEHOLDER cell**，跑到它會把 `PROJECT_ID` 蓋成字串 `"PLACEHOLDER"`，後面全炸。已移除。
4. **最後一格存檔時是失敗狀態**（`name Content is not defined`）。helper 已補齊 import 並加上防呆。

---

## 架構總覽

```
                        ┌─────────────────┐
                        │  使用者查詢 🗣️   │
                        └────────┬────────┘
                                 ▼
                        ┌─────────────────┐
                        │  router_agent 🧠 │   LlmAgent：只負責挑路，不解題
                        └────────┬────────┘
         ┌───────────────┬───────┴────────┬────────────────┐
         ▼               ▼                ▼                ▼
  ┌─────────────┐ ┌──────────────┐ ┌──────────────┐ ┌──────────────┐
  │ foodie      │ │ §2 串鏈       │ │ §3 回頭邊     │ │ §4 fan-out   │
  │ 單一 agent   │ │ Workflow     │ │ Workflow     │ │ Workflow     │
  └─────────────┘ └──────────────┘ └──────────────┘ └──────────────┘
                          │                │                │
        START→find→navigate      START→draft→gate    START→(a‖b‖c)
                                   ↑        │              →JoinNode
                                   └planner─┘                →synthesis
```

## 執行前須知

- **認證**：這本走 **Vertex AI**（企業情境）。需要 GCP 專案 ＋ `gcloud auth application-default login`
- **執行時間**：約 5–10 分鐘，主要花在 LLM 呼叫
- **圖形**：graph 用 graphviz 畫。macOS `brew install graphviz`／Ubuntu `apt install graphviz`／Colab 已內建

# 0️⃣ 設定與驗證 🔑

三件事：裝套件、設定 Vertex AI、準備整本共用的 helper。

> **關於 `PROJECT_ID`**
> 舊版直接把專案 ID 寫在 cell 裡，教材發出去就等於把自己的 GCP 專案 ID 一起發出去。
> 這一版改成：**環境變數優先** → 有互動前端才問 → 都沒有就明確告訴你要設什麼。

In [ ]:
# ── 安裝套件（第一次執行才需要，把 # 拿掉）─────────────────────────────
# !pip install -q "google-adk>=2.8,<3" google-genai
# 用 uv：
# !uv pip install -q "google-adk>=2.8,<3" google-genai
#
# graphviz 的 dot 執行檔要另外裝（畫 graph 用）：
#   macOS  : brew install graphviz
#   Ubuntu : sudo apt-get install -y graphviz
#   Colab  : 已內建

import importlib.metadata as _md
import shutil
import sys

print(f"Python           {sys.version.split()[0]}")
for pkg in ["google-adk", "google-genai", "google-cloud-aiplatform", "graphviz"]:
    try:
        print(f"{pkg:<26} {_md.version(pkg)}")
    except _md.PackageNotFoundError:
        print(f"{pkg:<26} （未安裝）")

print(f"\ngraphviz `dot`  {shutil.which('dot') or '❌ 找不到（graph 會退回文字模式，其他不受影響）'}")
print("本教材驗證版本：google-adk 2.8.0 / google-genai 2.20.0 / Python 3.14.7")

In [ ]:
# ── 匯入 ADK 2.0 需要的東西 ───────────────────────────────────────────────
import asyncio
import json
import logging
import os
import re
import warnings

from IPython.display import HTML, Markdown, display

# ── ADK 2.0 的編排主角：Workflow ───────────────────────────────────────
from google.adk.workflow import Workflow, node, START, JoinNode, DEFAULT_ROUTE
#   Workflow  : 用 edges 描述的 DAG，取代 Sequential/Loop/ParallelAgent
#   node      : 把普通 Python 函式包成 workflow 節點的 decorator
#   START     : 圖的進入點
#   JoinNode  : fan-in 專用 —— 等所有上游到齊、只執行一次
#   DEFAULT_ROUTE : routing map 的 fallback 分支

from google.adk.agents import Agent, Context, LlmAgent
#   Agent 是 LlmAgent 的別名（兩個都可以用，本教材統一用 Agent 比較短）
#   Context : 節點拿到的執行上下文 —— ctx.state / ctx.user_content / ctx.route

from google.adk.apps import App, ResumabilityConfig
from google.adk.events import Event
from google.adk.runners import Runner, InMemoryRunner
from google.adk.sessions import InMemorySessionService, Session
from google.adk.tools import google_search, ToolContext
from google.genai import types
from google.genai.types import Content, Part      # ← 舊版最後一格就是漏了這個而失敗

# ── 安靜的 log（只關掉已知無資訊量的噪音，不要用光禿禿的 filterwarnings("ignore")）──
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=r"\[EXPERIMENTAL\].*")
logging.getLogger("google_genai").setLevel(logging.ERROR)   # 關掉 AFC 的長篇建議
logging.getLogger("google_adk").setLevel(logging.ERROR)

print("✅ ADK 2.0 模組載入完成")
print(f"   Agent is LlmAgent 別名: {Agent is LlmAgent}")

In [ ]:
# ── Vertex AI 設定（PROJECT_ID 不寫死）───────────────────────────────────
# ADK 透過 google-genai 呼叫 Gemini，走哪一條由環境變數決定：
#   GOOGLE_GENAI_USE_VERTEXAI = "True"  → Vertex AI（要 GCP 專案 + ADC 認證）
#   GOOGLE_GENAI_USE_VERTEXAI = "0"     → AI Studio（只要一把 API key）
# 這本預設 Vertex AI；但**如果你已經自己設好了就沿用**，不覆蓋。


def can_prompt() -> bool:
    """現在的環境有沒有辦法跟人要輸入？

    JupyterLab / Colab → True；nbclient / papermill 這種 headless → False。
    （sys.stdin.isatty() 在三種情況都是 False，分不出來，不能用。）
    """
    try:
        return bool(get_ipython().kernel._allow_stdin)  # noqa: F821
    except Exception:
        return False


# 已經設過就不動它（讓這本 notebook 也能在 AI Studio 模式下被自動化執行）
USE_VERTEX = os.environ.get("GOOGLE_GENAI_USE_VERTEXAI")
if USE_VERTEX is None:
    USE_VERTEX = "True"
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = USE_VERTEX
VERTEX_MODE = str(USE_VERTEX).lower() in ("1", "true", "yes")

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")

if VERTEX_MODE and not PROJECT_ID and can_prompt():
    PROJECT_ID = input("請輸入你的 GCP PROJECT_ID: ").strip()

if VERTEX_MODE:
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")

# HAS_AUTH 決定後面要不要真的呼叫 LLM；沒認證的 cell 會自己印 SKIPPED 跳過，
# 不會讓整本執行中斷（graph 那些不用認證的部分照樣看得到）。
if VERTEX_MODE:
    ADC = os.path.expanduser("~/.config/gcloud/application_default_credentials.json")
    HAS_AUTH = bool(PROJECT_ID) and (os.path.isfile(ADC)
                                     or bool(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")))
    print(f"認證模式  Vertex AI")
    print(f"專案      {PROJECT_ID or '❌ 未設定'}")
    print(f"區域      {LOCATION}")
    print(f"ADC       {'✅ 已就緒' if os.path.isfile(ADC) else '❌ 找不到'}")
    if not HAS_AUTH:
        print("\n⚠️ Vertex AI 還沒準備好。請在終端機執行：")
        print(f"   gcloud auth application-default login --project=<你的專案ID>")
        print("   並確認已啟用 Vertex AI API：")
        print("   gcloud services enable aiplatform.googleapis.com --project=<你的專案ID>")
        print("\n   或者改用 AI Studio（門檻低很多）：")
        print('   os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "0"')
        print('   os.environ["GOOGLE_API_KEY"] = "<你的 key>"   # https://aistudio.google.com/apikey')
        print("   然後從這一格重新執行。")
else:
    HAS_AUTH = bool(os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY"))
    print(f"認證模式  AI Studio API key")
    print(f"API key   {'✅ 已設定' if HAS_AUTH else '❌ 未設定'}")

print(f"模型      {MODEL}")
print(f"\n需要 LLM 的 cell 會執行嗎：{'✅ 會' if HAS_AUTH else '⏭ 不會（自動跳過，不中斷）'}")

In [ ]:
# ── 整本共用的 helper ①：執行 agent / workflow 並顯示過程 ────────────────
session_service = InMemorySessionService()
MY_USER_ID = "adk_adventurer_001"


async def run_agent_query(agent, query: str, quiet: bool = False,
                          show_nodes: bool = True) -> dict:
    """跑一個 Agent 或 Workflow，回傳結果與事件。

    Args:
        agent      : Agent / Workflow 都可以（ADK 2.0 兩者都是 BaseNode）
        query      : 使用者查詢
        quiet      : True 時不印過程（router 只要答案時用）
        show_nodes : True 時印出每個節點的產出（Workflow 才有意義）
    Returns:
        {"text": 最終文字, "events": [...], "state": 最終 session state}
    """
    app_name = f"app_{agent.name}"
    # is_resumable=True 才會送出節點狀態快照 → 才能畫「上色版」的 graph
    app = App(name=app_name, root_agent=agent,
              resumability_config=ResumabilityConfig(is_resumable=True))
    runner = Runner(app=app, session_service=session_service)
    session = await session_service.create_session(app_name=app_name, user_id=MY_USER_ID)

    if not quiet:
        print(f"🚀 執行 {agent.name}：{query[:60]}{'…' if len(query) > 60 else ''}")

    events, final_text = [], ""
    try:
        async for ev in runner.run_async(
            user_id=MY_USER_ID,
            session_id=session.id,
            new_message=Content(role="user", parts=[Part(text=query)]),
        ):
            events.append(ev)
            path = getattr(getattr(ev, "node_info", None), "path", None)
            name = path.split("/")[-1] if path else ev.author
            text = " ".join(p.text or "" for p in (ev.content.parts if ev.content else "")).strip()

            if not quiet and show_nodes:
                if ev.output is not None:          # function node 的回傳值
                    out = json.dumps(ev.output, ensure_ascii=False, default=str)
                    print(f"   ⚙️  {name:<22} {out[:88]}")
                elif text:                          # agent node 的文字輸出
                    print(f"   💬 {name:<22} {len(text)} 字")
            if text:
                final_text = text
    except Exception as e:
        print(f"   ❌ {type(e).__name__}: {e}")
        return {"text": "", "events": events, "state": {}, "error": str(e)}

    sess = await session_service.get_session(
        app_name=app_name, user_id=MY_USER_ID, session_id=session.id)
    return {"text": final_text, "events": events, "state": dict(sess.state or {})}


print("✅ run_agent_query() 已定義")

In [ ]:
# ── 整本共用的 helper ②：把 graph 畫出來 ─────────────────────────────────
# 這是 ADK 2.0 最實用的東西之一：Workflow 有一張真正的圖，可以直接畫。
# 降級順序：本機 graphviz SVG → 純文字節點/邊列表（所以沒裝 dot 也不會壞）


def _svg_html(svg: str, max_width: int) -> str:
    """graphviz 產出的 SVG 帶死的 pt 尺寸，大圖會被切掉。改成隨欄寬縮放 + 可橫向捲動。"""
    svg = re.sub(r'(<svg[^>]*?)\swidth="[^"]*"', r"\1", svg, count=1)
    svg = re.sub(r'(<svg[^>]*?)\sheight="[^"]*"', r"\1", svg, count=1)
    svg = svg.replace("<svg", f'<svg style="width:100%;height:auto;max-width:{max_width}px"', 1)
    return f'<div style="overflow-x:auto;max-width:100%">{svg}</div>'


def show_graph(obj, agent_state=None, dark_mode=False, max_width=760):
    """畫出 ADK App / Workflow / Agent。

    Args:
        obj         : App、Workflow 或 Agent
        agent_state : {"nodes": {"節點名": {"status": 0-6}}}，用 node_status() 從事件取，
                      會把節點按執行狀態上色（綠=跑完、白=沒走到）
        dark_mode   : SVG 自己畫不透明底色，所以跟 Jupyter 主題無關，兩種都看得清
    Returns:
        實際用到的方式（"svg" 或 "text"）
    """
    from google.adk.cli.utils.graph_serialization import serialize_app_info
    from google.adk.cli.utils.graph_visualization import plot_workflow_graph

    app = obj if isinstance(obj, App) else App(name=getattr(obj, "name", "app"), root_agent=obj)
    info = serialize_app_info(app)
    try:
        # format="svg" 回傳 str；"png"/"pdf" 回傳 bytes；"dot" 回傳 DOT 原始碼
        svg = plot_workflow_graph(info, agent_state, format="svg", dark_mode=dark_mode)
        display(HTML(_svg_html(svg, max_width)))     # SVG 字串要用 HTML/SVG，不能用 Image
        return "svg"
    except Exception as ex:
        print(f"（graphviz 不可用：{type(ex).__name__} → 改印文字版）")
        gr = (info.get("root_agent") or {}).get("graph") or {}
        print("nodes:", [n.get("name") for n in gr.get("nodes", [])])
        for e in gr.get("edges", []):
            route = f"   [route={e['route']}]" if e.get("route") else ""
            print(f"  {e['from_node']['name']} → {e['to_node']['name']}{route}")
        return "text"


def node_status(result_or_events) -> dict:
    """從事件流蒐集節點狀態快照，餵給 show_graph(..., agent_state=...) 就會上色。

    前提是 App 要開 resumability（run_agent_query 已經幫你開了），
    否則 ev.actions.agent_state 永遠是 None、圖上每個節點都是白的。
    """
    events = (result_or_events.get("events", []) if isinstance(result_or_events, dict)
              else result_or_events)
    snap = {}
    for ev in events:
        st = getattr(getattr(ev, "actions", None), "agent_state", None)
        if st and "nodes" in st:
            snap.update(st["nodes"])
    return {"nodes": snap}


from google.adk.workflow._node_status import NodeStatus
STATUS_NAME = {s.value: s.name for s in NodeStatus}

print("✅ show_graph() / node_status() 已定義")
print("   NodeStatus:", STATUS_NAME)
print("   ⚠️ status 必須是 int 或 NodeStatus，傳字串會被靜默當成 INACTIVE（白色）")

# 1️⃣ 為什麼 `Workflow` 取代三劍客

## 舊版的三個類別各自只解一半的問題

`SequentialAgent` 只能串一條線、`ParallelAgent` 只能分岔、`LoopAgent` 只能重複。
真實流程常常是「三路並行完再匯合，然後依結果走不同分支，不合格就回頭重做」——
用三劍客拼起來會變成一堆巢狀，而且**沒有一張圖可以看**。

`Workflow` 直接讓你寫 DAG 的邊：

```python
Workflow(name="...", edges=[
    (START, a),                        # 進入點
    (a, (b, c, d)),                    # fan-out：a 做完，b/c/d 同時開跑
    ((b, c, d), join),                 # fan-in：三個都到齊才繼續（join 必須是 JoinNode）
    (join, decider),
    (decider, {"ok": done, "bad": a}), # 條件分支；"bad" 這條是回頭邊 → 形成迴圈
])
```

## 四個一定要知道的坑

**① fan-in 一定要用 `JoinNode`。**
如果 `((b, c, d), merge)` 的 `merge` 是普通 `@node` 函式，ADK 會當成三條獨立的邊，
於是 **`merge` 會被執行三次**（事件路徑 `merge@1`、`merge@2`、`merge@3`），而且不報錯。
`JoinNode` 才會「等全部到齊、只跑一次」，它的 output 會自動整理成
`{"上游節點名": 那個節點的輸出}`。

**② `JoinNode` 必須是同一個物件實例。**
`edges` 裡兩處各寫一次 `JoinNode(name="x")` 會建出兩個物件，驗證直接失敗：
`Duplicate node names found: ['x']`。要先 `x = JoinNode(name="x")` 再引用。

**③ 條件分支靠 `ctx.route`，不是 return 值。**
`return "ok"` 沒有用——那只是節點的 output。要寫 `ctx.route = "ok"`。
寫錯**不會報錯**，只印一行 warning，然後流程就默默斷在那個節點。現場 demo 最容易中。
另外 `ctx.route` **只能在 function node 裡設**，`LlmAgent` 不能直接設 route——
所以「LLM 判斷 → 決定分支」的正確寫法是：LlmAgent 用 `output_key` 把結論寫進 state，
再由一個小的 function node 讀 state 並設 `ctx.route`（§3 會用到）。

**④ `Workflow` 沒有 `max_iterations`。**
`LoopAgent` 有這個保護，`Workflow` 的回頭邊**沒有**。迭代次數要自己記在 `ctx.state` 裡數，
不然模型一直不達標就會無限迴圈。§3 示範怎麼守。

## 節點之間怎麼傳資料

| 方式 | 寫法 | 說明 |
|---|---|---|
| 共享狀態 | `ctx.state["k"] = v` | 最通用 |
| 參數自動綁定 | `def f(k: str)` | 預設 `parameter_binding='state'`，按**參數名**去 `ctx.state` 找 |
| LlmAgent 寫入 | `Agent(output_key="k")` | 該 agent 的最終回覆自動存進 `state["k"]` |
| LlmAgent 讀取 | `instruction="…{k}…"` | instruction 裡的 `{k}` 會被 `state["k"]` 替換 |
| 看上游輸出 | `ev.output` | function node 有；agent node 是 `None`，文字在 `ev.content` |

> ⚠️ **`{k}` 只吃單層 key。**
> `{session.query}` 這種帶點的名字**不會**被替換，會原封不動當文字送給模型（舊版教材的 bug）。
> 而 `{不存在的key}` 會直接丟 `KeyError: Context variable not found`；
> 想要「有就代入、沒有就空白」要寫 `{k?}`。

# 2️⃣ 順序執行：`SequentialAgent` → `Workflow` 串鏈

## 情境

使用者說：「幫我找 Palo Alto 最好的壽司，然後告訴我從 Caltrain 站怎麼過去。」
這需要兩種技能：找餐廳、規劃路線。而且**第二步要用到第一步的答案**。

## 三代寫法對照

**第一代（最原始）**：Python 手動串接。跑完 `foodie_agent`，用 regex 從回覆裡
撈出店名，再組一個新 query 丟給 `transportation_agent`。

```python
foodie_response = await run_agent_query(foodie_agent, query, ...)
match = re.search(r'\*\*(.*?)\*\*', foodie_response)     # ← 脆弱！
destination = match.group(1)
directions_query = f"Give me directions to {destination} from ..."
await run_agent_query(transportation_agent, directions_query, ...)
```

問題：那個 regex 綁死了「LLM 一定會用 `**粗體**` 標店名」的假設。
模型換個格式就抓不到，而且 `match` 是 `None` 時整段炸掉。

**第二代（舊版教材的答案）**：`SequentialAgent` ＋ `output_key`。
用 `output_key="destination"` 讓 ADK 自動把結果存進 state，
下一個 agent 的 instruction 寫 `{destination}` 就自動代入。regex 消失了。

**第三代（這一版）**：一樣用 `output_key` ＋ `{destination}`（這兩個**沒有** deprecated），
但編排從 `SequentialAgent` 換成 `Workflow` 的邊。好處是**畫得出圖**，
而且要加分支、加並行、加回頭邊時不用換類別。

另外前面多了一個 `capture_query` 節點，把原始使用者訊息抄進 `state["user_query"]`。
這是舊版 `{session.query}` 想做但做不到的事（ADK 不替換帶點的 placeholder），
也讓「從原始訊息找出發地」這件事變可靠——不然模型有時會回「請問您的出發地在哪裡呢？」。

In [ ]:
# ── §2 的節點 ────────────────────────────────────────────────────────────
@node
def capture_query(ctx: Context) -> dict:
    """把原始使用者訊息抄進 state，之後任何節點都能用 {user_query} 取用。

    ⭐ 這就是舊版教材 `{session.query}` 想做但做不到的事。
       ADK 的 placeholder 只吃**單層 state key**，`{session.query}` 帶點，
       不會被替換，會原封不動當文字送進 prompt。
       正確做法就是這樣：自己抄一份進 state，再用 {user_query}。

    為什麼需要它：Workflow 下游的 LlmAgent 雖然通常看得到 session 歷史，
    但「叫模型自己去歷史裡翻出發地」不可靠 —— 實測同一段 query 有時會回
    「請問您的出發地在哪裡呢？」。明確傳進 instruction 才穩。
    """
    q = " ".join(p.text or "" for p in (ctx.user_content.parts if ctx.user_content else "")).strip()
    ctx.state["user_query"] = q
    print(f"   📌 capture_query  原始訊息已存進 state（{len(q)} 字）")
    return {"user_query": q}


foodie_agent = Agent(
    name="foodie_agent",
    model=MODEL,
    tools=[google_search],
    description="美食評論家：找出最好的餐廳。",
    # instruction 要求「只輸出店名」—— 因為這個結果會直接被下一個節點當「目的地」用
    instruction=(
        "你是美食評論家。根據使用者的需求找出最好的餐廳。\n"
        "你必須**只輸出店名本身**，不要有任何其他文字、標點或說明。\n"
        "例如最好的壽司在 Jin Sho，你就只輸出：Jin Sho"
    ),
    output_key="destination",
    # ⭐ output_key：ADK 會自動把這個 agent 的最終回覆存進 state["destination"]
    #    取代了第一代那個脆弱的 regex
)

transportation_agent = Agent(
    name="transportation_agent",
    model=MODEL,
    tools=[google_search],
    description="導航助理：給出從起點到目的地的路線。",
    # ⭐ {destination} 是 state placeholder，執行前會被 state["destination"] 的值替換
    instruction=(
        "你是導航助理。\n"
        "使用者的原始訊息：{user_query}\n"      # ← 由 capture_query 存進 state
        "他要去的地方是：{destination}\n"        # ← 由 foodie_agent 的 output_key 存進 state
        "請從原始訊息裡找出出發地，然後給出從出發地到目的地的清楚路線建議。\n"
        "用繁體中文回答，200 字內。"
    ),
)

# ── 用 Workflow 的邊串起來 ───────────────────────────────────────────────
# 舊版：SequentialAgent(name=..., sub_agents=[foodie_agent, transportation_agent])
# 新版：把「誰接誰」寫成邊
find_and_navigate = Workflow(
    name="find_and_navigate",
    description="先找地點，再規劃到那裡的路線。",
    edges=[
        (START, capture_query),
        (capture_query, foodie_agent),
        (foodie_agent, transportation_agent),
    ],
)

print(f"✅ Workflow: {find_and_navigate.name}")
for e in find_and_navigate.graph.edges:
    print(f"   {e.from_node.name:>22} → {e.to_node.name}")

### 👀 把圖叫出來

圖上讀得到的東西：橢圓是 `START` / `END`、圓角矩形是 agent 節點、
虛線 🔧 是 agent 的 tool（`google_search` 會自動被畫出來，不用你標）。

In [ ]:
show_graph(find_and_navigate)

In [ ]:
# ── 跑起來 ───────────────────────────────────────────────────────────────
seq_result = None
if not HAS_AUTH:
    print("⏭ SKIPPED：沒有認證（上面的 graph 不需要認證，已經畫出來了）")
else:
    seq_result = await run_agent_query(
        find_and_navigate,
        "幫我找 Palo Alto 最好的壽司，然後告訴我從 Caltrain 車站怎麼過去。",
    )
    print(f"\n   state 裡的 destination = {seq_result['state'].get('destination')!r}")
    print(f"   ↑ 這是 foodie_agent 透過 output_key 自動寫進去的，"
          f"transportation_agent 用 {{destination}} 讀到它")
    display(Markdown(f"### 🍣🚗 最終路線\n\n{seq_result['text']}"))

In [ ]:
# ── 跑完的圖（用執行狀態上色）───────────────────────────────────────────
if seq_result:
    st = node_status(seq_result)
    print("節點狀態：", {k: STATUS_NAME.get(v.get("status")) for k, v in st["nodes"].items()})
    show_graph(find_and_navigate, agent_state=st)
else:
    print("⏭ SKIPPED")

# 3️⃣ 迭代改進：`LoopAgent` → `Workflow` 回頭邊

## `Workflow` 是 DAG，那怎麼做迴圈？

用 **routing map 的回頭邊**：讓某個分支指回上游的節點。實測會真的迴圈執行。

```python
Workflow(edges=[
    (START, draft), (draft, gate),
    (gate, {"refine": planner, "done": publish}),
    (planner, gate),                    # ← 回頭邊：planner 做完又回到 gate
])
```

## ⚠️ 兩個跟 `LoopAgent` 不一樣的地方

**① 沒有 `max_iterations`。**
`LoopAgent(max_iterations=3)` 幫你守上限；`Workflow` 的回頭邊**沒有任何保護**。
迭代次數要自己記在 `ctx.state` 裡數，並在閘門裡強制收手。不做的話，
模型一直不達標就是無限迴圈——這是這一節最重要的一句話。

**② 沒有 `escalate`。**
`LoopAgent` 是靠 tool 裡 `tool_context.actions.escalate = True` 跳出。
`Workflow` 改成宣告式：閘門把 `ctx.route` 設成 `"done"` 就走出去，設成 `"refine"` 就回頭。
比起「叫 LLM 記得呼叫 exit_loop tool」，這個做法不依賴模型自覺。

## 這一節的設計重點：**閘門是程式碼，創意才是 LLM**

舊版的 `critic_agent` 是個 LlmAgent，要它自己判斷「交通時間有沒有超過 45 分鐘」
並且在滿意時輸出一個約定好的句子。這有兩個問題：模型可能算錯距離，
也可能忘記輸出那句話——**迴圈跑幾輪變成看模型心情**。

這一版把判斷交給程式碼：

- `draft`（function node）給一個**刻意動線很差**的初版方案 → 第一關必定不合格
- `planner`（LlmAgent）負責**改進**方案（這是 LLM 真正擅長的事）
- `gate`（function node）解析方案 → 查交通時間表 → 決定回頭或放行

於是迴圈**至少會跑兩輪**，而且「為什麼要再跑一輪」有具體數字，不是模型的感覺。

> 💡 順帶修掉舊版的 bug：舊版 `refiner_agent` 的 instruction 寫了 `{session.query}`
> 想拿到原始查詢，但 ADK 不會替換帶點的名字——那 15 個字元原封不動進了 prompt。
> 要傳原始查詢就自己寫進 state（`ctx.state["user_query"] = ...`）再用 `{user_query}`。

In [ ]:
# ── 交通時間表：假資料，但是「確定的」 ──────────────────────────────────
# 教學重點是編排，不是地圖 API。用固定的經緯度算出確定的分鐘數，
# 這樣每次執行的行為都一致，不會因為外部 API 或模型而飄。
MAX_ROUNDS = 4        # ⚠️ 自製的迭代上限（Workflow 沒有內建 max_iterations）
LIMIT_MIN = 25        # 動線上限：超過就要求重做

VENUES = {
    # 活動候選
    "探索博物館": (37.801, -122.397),
    "笛洋美術館": (37.771, -122.469),
    "SFMOMA":     (37.786, -122.401),
    "金門公園":   (37.769, -122.483),
    # 餐廳候選
    "La Mar":      (37.799, -122.398),
    "Nopa":        (37.775, -122.437),
    "Zuni Cafe":   (37.777, -122.422),
    "Swan Oyster": (37.791, -122.421),
}


def travel_minutes(a: str, b: str) -> int:
    """兩個場館之間的交通時間（分鐘）。用經緯度直線距離換算，確定且可重現。"""
    (la1, lo1), (la2, lo2) = VENUES[a], VENUES[b]
    km = ((la1 - la2) ** 2 + (lo1 - lo2) ** 2) ** 0.5 * 111
    return round(km * 4 + 4)


def parse_plan(plan: str):
    """從「活動: X | 餐廳: Y」抽出兩個場館名。抽不到回 (None, None)。"""
    head, _, tail = plan.partition("餐廳")
    return (next((v for v in VENUES if v in head), None),
            next((v for v in VENUES if v in tail), None))


print("✅ 交通時間表已定義。幾個例子：")
for a, b in [("金門公園", "La Mar"), ("探索博物館", "La Mar"), ("SFMOMA", "Zuni Cafe")]:
    print(f"   {a} → {b}: {travel_minutes(a, b)} 分鐘"
          f"  {'❌ 超過上限' if travel_minutes(a, b) > LIMIT_MIN else '✅ 合格'}")

In [ ]:
# ── §3 的四個節點 ────────────────────────────────────────────────────────
@node
def draft(ctx: Context) -> dict:
    """初版草稿：刻意給一組動線很差的組合。

    模擬真實情境：初版方案來自使用者、舊系統或粗略估計，agent 的工作是改進它。
    因為這個初版是寫死的，第一關**必定**不合格 → 迴圈一定至少跑兩輪，
    教學示範不會因為模型剛好猜對而看不到迭代。
    """
    plan = "活動: 金門公園 | 餐廳: La Mar"
    ctx.state["current_plan"] = plan
    ctx.state["round"] = 0
    ctx.state["criticism"] = "（初版，尚未審查）"
    print(f"   📝 draft    初版：{plan}（實際 {travel_minutes('金門公園', 'La Mar')} 分鐘）")
    return {"plan": plan}


planner = Agent(
    name="planner",
    model=MODEL,
    output_key="current_plan",          # 新方案覆蓋 state["current_plan"]
    description="行程規劃師：根據審查意見改進方案。",
    instruction=(
        "你是舊金山行程規劃師，任務是**改進**現有方案。\n"
        "只能從下面挑，不要自己發明場館：\n"
        "  活動候選：探索博物館 / 笛洋美術館 / SFMOMA / 金門公園\n"
        "  餐廳候選：La Mar / Nopa / Zuni Cafe / Swan Oyster\n\n"
        "目前方案：{current_plan}\n"
        "審查意見：{criticism}\n\n"
        "請提出一組**動線更短**的新組合。\n"
        "只輸出一行，嚴格照這個格式，不要有任何其他文字：\n"
        "活動: <名稱> | 餐廳: <名稱>"
    ),
)


@node
def gate(ctx: Context) -> dict:
    """閘門：解析方案 → 查交通時間 → 決定回頭或放行。

    ⚠️ 兩個關鍵：
      1. ctx.route 只能在 function node 裡設，LlmAgent 設不了 —— 所以「LLM 判斷 →
         決定分支」的正確寫法是 LlmAgent 用 output_key 寫 state，再由這種節點讀 state 設 route。
      2. n > MAX_ROUNDS 的強制收手是必要的。Workflow 沒有 max_iterations，
         少了這一行、模型又一直不達標，就是無限迴圈。
    """
    n = ctx.state.get("round", 0) + 1
    ctx.state["round"] = n
    activity, restaurant = parse_plan(ctx.state.get("current_plan") or "")

    if not activity or not restaurant:
        ctx.state["criticism"] = "看不懂格式，請嚴格用「活動: X | 餐廳: Y」，場館要從候選清單挑。"
        ctx.route = "done" if n > MAX_ROUNDS else "refine"
        print(f"   🚦 gate     第 {n} 關：解析失敗 → route={ctx.route}")
        return {"round": n, "route": ctx.route}

    mins = travel_minutes(activity, restaurant)
    ctx.state["travel_minutes"] = mins
    passed = mins <= LIMIT_MIN
    hit_cap = n > MAX_ROUNDS
    ctx.route = "done" if (passed or hit_cap) else "refine"
    ctx.state["criticism"] = (
        "動線合格。" if passed else
        f"「{activity}」到「{restaurant}」要 {mins} 分鐘，超過 {LIMIT_MIN} 分鐘上限，請換更近的一組。"
    )
    flag = "✅ 通過" if passed else ("⛔ 達上限，強制收手" if hit_cap else "❌ 太遠，退回重做")
    print(f"   🚦 gate     第 {n} 關：{activity} → {restaurant} = {mins} 分鐘  "
          f"{flag} → route={ctx.route}")
    return {"round": n, "activity": activity, "restaurant": restaurant,
            "minutes": mins, "route": ctx.route}


@node
def publish(ctx: Context) -> dict:
    """定案。"""
    print(f"   🎯 publish  定案：{ctx.state.get('current_plan')}"
          f"（{ctx.state.get('travel_minutes')} 分鐘，共審查 {ctx.state.get('round')} 關）")
    return {"final_plan": ctx.state.get("current_plan"),
            "minutes": ctx.state.get("travel_minutes"),
            "rounds": ctx.state.get("round")}


# ── 組成帶回頭邊的 Workflow ─────────────────────────────────────────────
iterative_planner = Workflow(
    name="iterative_planner",
    description="給一個粗糙初版，讓 agent 反覆改進到動線合格為止。",
    edges=[
        (START, draft),
        (draft, gate),
        (gate, {"refine": planner, "done": publish}),   # 條件分支
        (planner, gate),                                 # ⭐ 回頭邊 → 形成迴圈
    ],
)

print(f"\n✅ Workflow: {iterative_planner.name}")
for e in iterative_planner.graph.edges:
    route = f"   [route={e.route}]" if e.route else ""
    back = "   ← 回頭邊" if e.from_node.name == "planner" else ""
    print(f"   {e.from_node.name:>10} → {e.to_node.name:<10}{route}{back}")

### 👀 圖上看得到迴圈

`gate` 是**菱形**（因為它有條件分支），兩條出邊標著 `route=refine` / `route=done`。
`planner → gate` 那條把圖接回去，迴圈在圖上一目了然——這是 `LoopAgent` 做不到的事。

In [ ]:
show_graph(iterative_planner)

In [ ]:
# ── 跑起來（會看到多輪迭代）─────────────────────────────────────────────
loop_result = None
if not HAS_AUTH:
    print("⏭ SKIPPED：沒有認證")
else:
    loop_result = await run_agent_query(
        iterative_planner,
        "幫我規劃舊金山一日遊，要有博物館和晚餐，動線越短越好。",
        show_nodes=False,      # 節點自己會 print，不用重複印
    )
    s = loop_result["state"]
    display(Markdown(f"""### 🔁 迭代結果

| | |
|---|---|
| 最終方案 | `{s.get('current_plan')}` |
| 動線 | {s.get('travel_minutes')} 分鐘（上限 {LIMIT_MIN}） |
| 審查關數 | {s.get('round')} |
| 最後的審查意見 | {s.get('criticism')} |

初版是寫死的 44 分鐘方案，所以第一關必定退回；
中間那一步「換成更近的組合」是 `planner` 這個 LlmAgent 真的做的。
"""))

In [ ]:
# ── 跑完的圖 ─────────────────────────────────────────────────────────────
if loop_result:
    st = node_status(loop_result)
    print("節點狀態：", {k: STATUS_NAME.get(v.get("status")) for k, v in st["nodes"].items()})
    show_graph(iterative_planner, agent_state=st)
else:
    print("⏭ SKIPPED")

# 4️⃣ 並行執行：`ParallelAgent` → fan-out ＋ `JoinNode`

## 情境

「幫我找一間博物館、一場音樂會，還有一家餐廳。」三件事**互不相依**，
一個一個跑要三倍時間，同時跑只要一倍（瓶頸是最慢的那個）。

## 舊版要包兩層，新版一張圖畫完

舊版的 `ParallelAgent` 只管「同時跑」，**沒有 fan-in 的概念**。
所以要拿到彙整結果，得在外面再包一層 `SequentialAgent`：

```python
parallel_research = ParallelAgent(sub_agents=[museum, concert, restaurant])
parallel_planner  = SequentialAgent(sub_agents=[parallel_research, synthesis])
#                   ↑ 為了「等三個跑完再整合」而存在的第二層
```

`Workflow` 有 `JoinNode`，所以 fan-out、fan-in、整合可以寫在**同一張圖**裡：

```python
Workflow(edges=[
    (START, (museum, concert, restaurant)),          # fan-out
    ((museum, concert, restaurant), gather),         # fan-in（gather 是 JoinNode）
    (gather, synthesis),
])
```

> ⚠️ 三個並行的 agent **必須用不同的 `output_key`**，不然會互相覆蓋。
> 這一點新舊版都一樣。

In [ ]:
# ── 三個互不相依的專家 agent ─────────────────────────────────────────────
# 每個都有自己的 output_key，不會互相覆蓋
museum_agent = Agent(
    name="museum_agent", model=MODEL, tools=[google_search],
    description="博物館專家。",
    instruction="你是博物館專家。依使用者需求找出一間最推薦的博物館。只輸出館名，不要其他文字。",
    output_key="museum_result",
)
concert_agent = Agent(
    name="concert_agent", model=MODEL, tools=[google_search],
    description="活動指南。",
    instruction="你是活動指南。依使用者需求找出一場音樂會或表演。只輸出「表演者 @ 場地」，不要其他文字。",
    output_key="concert_result",
)
restaurant_agent = Agent(
    name="restaurant_agent", model=MODEL, tools=[google_search],
    description="美食評論家。",
    instruction="你是美食評論家。依使用者需求找出一家最推薦的餐廳。只輸出店名，不要其他文字。",
    output_key="restaurant_result",
)

# ⭐ fan-in 一定要 JoinNode，而且只能建一個實例（下面兩條邊都引用同一個 gather）
gather = JoinNode(name="gather")

# 整合 agent：三個 placeholder 對應三個 output_key
synthesis_agent = Agent(
    name="synthesis_agent", model=MODEL,
    description="把三份研究結果整理成一份摘要。",
    instruction=(
        "把下面三份研究結果整理成一份簡潔的繁體中文條列摘要，每項一行，並加一句總結：\n"
        "- 博物館：{museum_result}\n"
        "- 音樂會：{concert_result}\n"
        "- 餐廳：{restaurant_result}"
    ),
)

parallel_planner = Workflow(
    name="parallel_planner",
    description="同時研究三件互不相依的事，匯合後整合成一份摘要。",
    edges=[
        (START, (museum_agent, concert_agent, restaurant_agent)),        # fan-out
        ((museum_agent, concert_agent, restaurant_agent), gather),       # fan-in
        (gather, synthesis_agent),
    ],
    max_concurrency=3,      # 同時最多跑 3 個節點（None = 不限）
)

print(f"✅ Workflow: {parallel_planner.name}")
for e in parallel_planner.graph.edges:
    print(f"   {e.from_node.name:>20} → {e.to_node.name}")

In [ ]:
show_graph(parallel_planner)

In [ ]:
# ── 跑起來 ───────────────────────────────────────────────────────────────
import time

par_result = None
if not HAS_AUTH:
    print("⏭ SKIPPED：沒有認證")
else:
    t0 = time.time()
    par_result = await run_agent_query(
        parallel_planner,
        "幫我規劃舊金山之旅，我需要一間博物館、一場音樂會，還有一家很棒的餐廳。",
    )
    elapsed = time.time() - t0
    s = par_result["state"]
    print(f"\n   總耗時 {elapsed:.1f}s（三個 agent 同時跑，所以 ≈ 最慢那個，不是三者相加）")
    print(f"   三個 output_key 各自的值：")
    for k in ("museum_result", "concert_result", "restaurant_result"):
        print(f"      {k:<20} {str(s.get(k))[:60]}")
    display(Markdown(f"### ⚡ 整合結果\n\n{par_result['text']}"))

In [ ]:
# ── 跑完的圖：三路並行 + JoinNode 匯合，全部變綠 ────────────────────────
if par_result:
    st = node_status(par_result)
    print("節點狀態：", {k: STATUS_NAME.get(v.get("status")) for k, v in st["nodes"].items()})
    show_graph(parallel_planner, agent_state=st)
else:
    print("⏭ SKIPPED")

### 💡 `JoinNode` 到底做了什麼

兩件事：

1. **等**——三個上游都完成才觸發下游，只觸發一次。
   如果 `gather` 換成普通 `@node` 函式，它會被跑三次（每條入邊各一次），而且不報錯。
2. **收**——它的 output 自動整理成 `{"上游節點名": 那個節點的輸出}`。
   而且 **`JoinNode` 下游的 `LlmAgent` 會自動收到這份匯合結果當輸入**，
   所以就算不用 `{museum_result}` 這些 placeholder，`synthesis_agent` 也看得到三份結果。
   這裡兩種都用上了：placeholder 讓 instruction 更明確，自動輸入當保險。

# 5️⃣ Router：把三種 `Workflow` 當成 worker

## Router 模式沒有被 deprecated

`router_agent` 就是一個普通 `LlmAgent`：它不解題，只回答「這題該給誰做」。
這個模式在 ADK 2.0 完全沒變，而且因為 **`Workflow` 也是 `BaseNode`**，
對 router 來說 Workflow 跟一般 agent 用起來一模一樣——都是 dict 查表然後跑。

```python
worker_agents = {
    "foodie_agent":       foodie_agent,        # 單一 Agent
    "find_and_navigate":  find_and_navigate,   # §2 的 Workflow
    "iterative_planner":  iterative_planner,   # §3 的 Workflow（含回頭邊）
    "parallel_planner":   parallel_planner,    # §4 的 Workflow（含 JoinNode）
}
```

dispatch 邏輯就一行：`if route in worker_agents: 跑它`。

## 為什麼 router 用 LLM，閘門用程式碼

這本教材兩種都示範了，差別值得記住：

| | router（§5） | gate（§3） |
|---|---|---|
| 誰做決定 | LlmAgent | function node |
| 為什麼 | 「使用者想幹嘛」是語意問題，規則寫不完 | 「44 分鐘有沒有超過 25」是算術，不該問模型 |
| 錯了會怎樣 | 選錯 worker，答案不對但不會壞 | 如果交給模型，迴圈跑幾輪變成看運氣 |

**能用程式碼判斷的就別問模型**——省 token、可重現、好 debug。

In [ ]:
# ── Router agent ─────────────────────────────────────────────────────────
# instruction 要把每個選項的「適用 / 不適用」寫清楚，router 的準確度全看這段
router_agent = Agent(
    name="router_agent",
    model=MODEL,
    description="分派員：判斷查詢該交給哪個 agent 或 workflow。",
    instruction="""你是請求分派員。分析使用者的查詢，決定該交給下面哪一個處理。
不要自己回答問題，只輸出最適合的那一個名稱。

可選項目：
- 'foodie_agent'：**只**問吃什麼、餐廳推薦，沒有其他需求。
- 'find_and_navigate'：要求「先找一個地點」**然後**「告訴我怎麼過去」的複合查詢。
- 'iterative_planner'：要規劃行程，而且有需要反覆檢查的**約束條件**（例如動線要短、預算上限）。
- 'parallel_planner'：一次要找**多個互不相干**的東西（例如博物館 AND 音樂會 AND 餐廳）。

只輸出一個名稱，不要有引號、不要有其他任何文字。""",
)

# ⭐ Workflow 和 Agent 混在同一個 dict 裡 —— 對 router 完全透明
worker_agents = {
    "foodie_agent": foodie_agent,
    "find_and_navigate": find_and_navigate,
    "iterative_planner": iterative_planner,
    "parallel_planner": parallel_planner,
}

print("✅ Router 已就緒，可分派的 worker：")
for name, w in worker_agents.items():
    kind = "Workflow" if isinstance(w, Workflow) else "Agent"
    n_nodes = f"（{len(w.graph.nodes)} 個節點）" if isinstance(w, Workflow) else ""
    print(f"   {name:<20} {kind:<9}{n_nodes}")

In [ ]:
# ── 完整流程：router 分派 → worker 執行 ─────────────────────────────────
async def dispatch(query: str):
    """問 router 該走哪條路，然後跑那個 worker。"""
    print(f"\n{'=' * 68}\n🗣️  {query}\n{'=' * 68}")

    # 步驟 1：router 決策（quiet=True，只要答案不要過程）
    r = await run_agent_query(router_agent, query, quiet=True)
    route = (r["text"] or "").strip().strip("'\"`").strip()
    print(f"🚦 router 選擇：{route!r}")

    # 步驟 2：統一 dispatch —— Workflow 和 Agent 都一樣跑
    worker = worker_agents.get(route)
    if worker is None:
        # router 給了不認識的名字，通常是 instruction 寫得不夠精確
        print(f"🚨 router 選了不存在的選項：{route!r}（可選：{list(worker_agents)}）")
        return None

    kind = "Workflow" if isinstance(worker, Workflow) else "Agent"
    print(f"--- 交給 {worker.name}（{kind}）---")
    result = await run_agent_query(worker, query)
    print(f"--- {worker.name} 完成 ---")
    return result


if not HAS_AUTH:
    print("⏭ SKIPPED：沒有認證")
else:
    queries = [
        "我想吃 Palo Alto 最好的壽司。",                                    # → foodie_agent
        "幫我找 Palo Alto 最好的壽司，然後告訴我從 Caltrain 站怎麼走過去。",   # → find_and_navigate
        "幫我規劃舊金山一日遊，博物館加晚餐，但兩個地點之間的交通時間要很短。",  # → iterative_planner
        "幫我規劃舊金山之旅，我需要一間博物館、一場音樂會、一家餐廳。",         # → parallel_planner
    ]
    dispatch_results = []
    for q in queries:
        dispatch_results.append(await dispatch(q))
    print(f"\n{'=' * 68}\n✅ 四種查詢全部分派完成")

In [ ]:
# ── 最後一眼：四個 worker 的圖並排看 ────────────────────────────────────
# 同一個 show_graph()，四種不同的編排形狀
for name in ["find_and_navigate", "iterative_planner", "parallel_planner"]:
    print(f"\n{'─' * 68}\n{name}\n{'─' * 68}")
    show_graph(worker_agents[name], max_width=620)

# 6️⃣ ADK 1.x → 2.0 遷移對照表

手上有舊教材或舊專案的話，這張表就是全部要改的東西。

## 編排

| ADK 1.x | ADK 2.0 | 備註 |
|---|---|---|
| `SequentialAgent(sub_agents=[a, b])` | `Workflow(edges=[(START, a), (a, b)])` | 1.x 寫法還能跑，但會噴 `DeprecationWarning` |
| `ParallelAgent(sub_agents=[a, b, c])` | `Workflow(edges=[(START, (a, b, c))])` | 同上 |
| `ParallelAgent` ＋ 外包 `SequentialAgent` 接整合 | 一張圖：fan-out ＋ `JoinNode` ＋ 整合 | 不用再包第二層 |
| ❌ 沒有 fan-in | `JoinNode` | **fan-in 目標一定要是 `JoinNode`**，普通節點會被跑 N 次且不報錯 |
| `LoopAgent(sub_agents=[...], max_iterations=3)` | routing map 回頭邊 `(gate, {"refine": planner, "done": publish})` ＋ `(planner, gate)` | ⚠️ **沒有 `max_iterations`，要自己在 `ctx.state` 數** |
| `tool_context.actions.escalate = True` 跳出迴圈 | 閘門把 `ctx.route` 設成 `"done"` | 不依賴模型記得呼叫 exit tool |
| ❌ 沒有條件分支 | `(node, {"a": x, "b": y})` ＋ `ctx.route = "a"` | **`ctx.route` 只能在 function node 設**；設錯只 warning 不報錯 |
| ❌ | `Workflow(max_concurrency=N)` | 限制同時執行的節點數 |
| ❌ | `node(fn, retry_config=…, timeout=…)` | 節點層級的重試與逾時 |

## 沒有變的東西（好消息）

| 東西 | 狀態 |
|---|---|
| `Agent` / `LlmAgent`、`instruction`、`tools` | ✅ 完全沒變 |
| `output_key` ＋ `{key}` placeholder | ✅ 完全沒變，`Workflow` 裡照用 |
| `google_search`、`FunctionTool`、`AgentTool`、`ToolContext` | ✅ 完全沒變 |
| `Runner`、`InMemorySessionService`、`Session`、`Event` | ✅ 完全沒變 |
| Router 模式（LlmAgent 當分派員） | ✅ 完全沒變 |

## 圖形（2.0 新增）

```python
from google.adk.apps import App, ResumabilityConfig
from google.adk.cli.utils.graph_serialization import serialize_app_info
from google.adk.cli.utils.graph_visualization import plot_workflow_graph

app = App(name="x", root_agent=wf,
          resumability_config=ResumabilityConfig(is_resumable=True))   # ← 要上色就必須開
svg = plot_workflow_graph(serialize_app_info(app), agent_state, format="svg", dark_mode=False)
```

- `format="svg"` / `"dot"` 回傳 **str**；`"png"` / `"pdf"` / `"jpg"` 回傳 **bytes**
- `agent_state` 形狀：`{"nodes": {"節點名": {"status": 0-6}}}`
- `status` 必須是 **int 或 `NodeStatus`**；傳字串會被**靜默**當成 `INACTIVE`（白色）
- 節點名是**裸名**（`gate`），不是事件路徑（`iterative_planner@1/gate@1`）
- 不開 `is_resumable=True` 的話 `ev.actions.agent_state` 永遠是 `None`，整張圖都是白的
- 1.x 的 `cli.utils.agent_graph.get_agent_graph()` 在 2.x **已經不存在**

## state placeholder 的三種行為（舊版教材踩過的坑）

| 寫法 | 行為 |
|---|---|
| `{current_plan}` | ✅ 被 `state["current_plan"]` 替換 |
| `{session.query}` | ⚠️ **不會替換**，原封不動當文字送給模型（帶點的名字 ADK 不處理） |
| `{does_not_exist}` | ❌ 執行時丟 `KeyError: Context variable not found` |
| `{does_not_exist?}` | ✅ 選擇性語法，取不到就代入空字串 |

# 7️⃣ 🎉 完成了

## 你學到的四種編排

| 編排 | Workflow 寫法 | 什麼時候用 |
|---|---|---|
| **串鏈** | `(START, a), (a, b)` | 後一步要用到前一步的結果 |
| **回頭邊** | `(gate, {"refine": p, "done": q}), (p, gate)` | 有品質約束、要反覆改到達標 |
| **fan-out ＋ JoinNode** | `(START, (a,b,c)), ((a,b,c), join), (join, s)` | 多個互不相依的子任務 |
| **Router** | LlmAgent 回名稱 ＋ dict 查表 | 一個入口要服務多種意圖 |

## 自我檢核

- [ ] 我能說出為什麼 fan-in 一定要 `JoinNode`（不用會怎樣？）
- [ ] 我記得條件分支是 `ctx.route = "..."`，**不是** `return "..."`
- [ ] 我知道 `ctx.route` 只能在 function node 設，LlmAgent 設不了
- [ ] 我知道 `Workflow` 的回頭邊**沒有** `max_iterations`，上限要自己守
- [ ] 我能說出 `output_key` 和 `{key}` 是怎麼配對的
- [ ] 我知道 `{session.query}` 這種帶點的 placeholder 不會被替換
- [ ] 我知道要畫上色的圖必須開 `ResumabilityConfig(is_resumable=True)`
- [ ] 我能判斷一個決策該給 LlmAgent 還是給 function node

## 下一步練習

1. **加第五種 worker**：做一個「預算約束」的 `iterative_planner` 變體，
   閘門檢查總花費而不是交通時間（提示：改 `gate` 就好，圖的形狀一樣）
2. **並行 ＋ 迴圈組合**：fan-out 三個方案 → JoinNode → 閘門挑最好的 →
   不夠好就整批重來（一張圖裡同時有 `JoinNode` 和回頭邊）
3. **加 `DEFAULT_ROUTE`**：讓 routing map 有 fallback 分支，
   這樣 route 值沒對上時不會默默斷掉（圖上那個 `⚠️ [NO DEFAULT]` 標記就會消失）
4. **節點層級的重試**：把 `planner` 換成 `node(planner, retry_config=RetryConfig(...))`
5. **換 session backend**：`InMemorySessionService` → `DatabaseSessionService`，讓對話跨重啟保留

---

## 相關教材

- 同資料夾的 `ADK_Learning_tools.ipynb` — 工具與記憶（`FunctionTool` / `AgentTool` / session）。
  那本用的 API 在 ADK 2.0 **完全沒有 deprecated**，不需要改寫。
- `stockpulse-ai/StockPulse_AI_ADK2_完整實戰.ipynb` — 把 `Workflow` 用在真實專案，
  加上真正的 A2A Protocol、LangGraph 橋接與 Cloud Run 部署。

```
  (\__/)
  (•ㅅ•)
  /づ  📚   享受學習 AI Agents 的樂趣 :)
```